# Chapter 10. CNN 기초와 응용 — 10.3 탐지·분할·전이학습 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter10_3_detection_segmentation_transfer.ipynb)

책 본문: [10.3 분류를 넘어서: 탐지, 분할, 전이학습](https://smhanlab.com/book-ml/kor/ml1/chapter10/3.html)

탐지 모델의 박스 중복 제거(IoU + NMS)를 직접 구현하고, 10.2의 `SmallCNN`
위에서 "스크래치 vs 특징 추출(동결) vs 미세조정" 세 팔(arm)을 비교하고,
"다른 작업"에서 사전학습한 가중치를 가져오는 두 단계 전이학습을 실행
확인합니다 — 전부 로컬에서 `python`으로 먼저 실행 검증했습니다.

## 1. IoU: 박스의 겹침을 재는 법

본문의 수식 그대로 — 교집합 면적 / 합집합 면적. \(\text{IoU}(A,B) =
|A \cap B| / |A \cup B|\).

In [1]:
def iou(a, b):
    """a, b = [x0, y0, x1, y1]. 교집합/합집합 면적 비율."""
    inter = max(0, min(a[2], b[2]) - max(a[0], b[0])) * \
            max(0, min(a[3], b[3]) - max(a[1], b[1]))
    union = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / union

A, B, C, D = [0,0,10,10], [1,1,11,11], [20,0,30,10], [21,1,31,11]
print(f"IoU(A,B) = {iou(A,B):.4f}  (손계산: 81/119 = {81/119:.4f})")
print(f"IoU(C,D) = {iou(C,D):.4f}")
print(f"IoU(A,C) = {iou(A,C):.4f}  (서로 멀리 떨어져 0)")
assert abs(iou(A, B) - 81/119) < 1e-12   # 본문의 손계산과 정확히 일치
assert iou(A, C) == 0.0

IoU(A,B) = 0.6807  (손계산: 81/119 = 0.6807)
IoU(C,D) = 0.6807
IoU(A,C) = 0.0000  (서로 멀리 떨어져 0)


## 2. NMS: 점수 높은 박스를 살리고 겹치는 박스를 제거

본문의 4개 박스 표(A 0.90, B 0.80, C 0.75, D 0.60)에 임계값 0.5의 NMS를
적용하면 **A와 C**만 남는다는 것을 확인합니다 — A-B, C-D 두 쌍의 IoU가
동일(0.68)이라, "제거 기준은 점수가 아닌 겹기"임을 보여줍니다.

In [2]:
def nms(scores, boxes, thr=0.5):
    """점수 내림차순으로, 임계값 thr를 넘는 박스끼리 겹치면 점수 낮은 쪽을 제거."""
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    keep, suppressed = [], set()
    for i in order:
        if i in suppressed:
            continue
        keep.append(i)
        for j in order:
            if j != i and j not in suppressed and iou(boxes[i], boxes[j]) > thr:
                suppressed.add(j)
    return keep

scores = [0.9, 0.8, 0.75, 0.6]
boxes  = [A, B, C, D]
kept = nms(scores, boxes)
print("NMS(0.5) keep =", kept, " ->", ["ABCD"[i] for i in kept])
assert kept == [0, 2]   # A와 C만 남는다 (본문 표와 일치)

NMS(0.5) keep = [0, 2]  -> ['A', 'C']


NMS가 두 쌍을 각각 "점수 높은 쪽 보관, 낮은 쪽 억제"하는 과정을 그립니다
— 검은 색이 교집합(81)이고, 0.68 > 0.5 임계값이라 점수 낮은 박스가
제거됩니다. (같은 그림을 `kor/src/images/ch10_3_iou_nms.svg`로 저장.)

In [3]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)

def box_rect(ax, b, color, alpha=0.30, lw=2.0, edge=None, zorder=2):
    x0, y0, x1, y1 = b
    ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, facecolor=color, alpha=alpha,
                               edgecolor=edge or color, linewidth=lw, zorder=zorder))

def pair_panel(ax, b_keep, b_sup, s_keep, s_sup):
    inter = [max(b_keep[0], b_sup[0]), max(b_keep[1], b_sup[1]),
             min(b_keep[2], b_sup[2]), min(b_keep[3], b_sup[3])]
    box_rect(ax, b_sup, "tab:orange", alpha=0.25, zorder=2)
    box_rect(ax, b_keep, "tab:blue", alpha=0.25, zorder=2)
    box_rect(ax, inter, "#212529", alpha=0.55, lw=0, zorder=3)
    box_rect(ax, b_keep, "none", lw=3, edge="green", zorder=4)
    ax.text(b_keep[2]+0.4, b_keep[3], f"Kept ({s_keep})", fontsize=9.5, color="green", va="center", fontweight="bold")
    ax.text(b_sup[0]-0.4, b_sup[1], f"Suppressed ({s_sup})", fontsize=9.5, color="darkorange", va="center", ha="right", fontweight="bold")
    ax.set_xlim(-1, 13); ax.set_ylim(-1, 12); ax.set_aspect("equal"); ax.axis("off")

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
pair_panel(axes[0], A, B, "A 0.90", "B 0.80")
axes[0].set_title("A-B pair: IoU ≈ 0.68", fontsize=10.5)
pair_panel(axes[1], C, D, "C 0.75", "D 0.60")
axes[1].set_title("C-D pair: IoU ≈ 0.68", fontsize=10.5)
for ax in axes:
    ax.text(5, -0.7, "Black = intersection (81); 0.68 > 0.5 threshold, so the lower-scoring box is removed",
            fontsize=8.5, color="#404040", ha="center", va="top")
fig.suptitle("NMS (threshold 0.5): 4 candidate boxes → 2 objects (only A and C remain)", fontsize=11.5)
fig.tight_layout(rect=[0, 0.02, 1, 0.94])
fig.savefig(IMG + "/ch10_3_iou_nms.svg")
plt.show()

/tmp/ipykernel_812454/750390293.py:35: UserWarning: Tight layout not applied. tight_layout cannot make Axes width small enough to accommodate all Axes decorations
  fig.tight_layout(rect=[0, 0.02, 1, 0.94])


## 3. 전이학습 세 팔 비교: 스크래치 vs 특징 추출 vs 미세조정

10.2의 `SmallCNN`(세로줄/가로줄 16×16 분류)을 그대로 쓰고, **동일
아키텍처·동일 데이터(300 학습/200 검증)·동일 30 에폭**에 *다만 어느 층이
갱신되는가*를 바꿉니다. 먼저 파라미터 장부 — 본문의 표와 일치해야 합니다.

In [4]:
import torch, torch.nn as nn

class SmallCNN(nn.Module):
    """10.2절의 SmallCNN을 그대로 재사용."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(16*4*4, 2)
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

m = SmallCNN()
p_all  = sum(p.numel() for p in m.parameters())
p_conv = sum(p.numel() for p in m.conv1.parameters()) + sum(p.numel() for p in m.conv2.parameters())
p_fc   = sum(p.numel() for p in m.fc.parameters())
print(f"전체 파라미터 {p_all} = 합성곱 {p_conv} ({p_conv/p_all:.0%}) + 분류층 {p_fc}")
assert (p_all, p_conv, p_fc) == (1762, 1248, 514)   # 본문 표와 일치

전체 파라미터 1762 = 합성곱 1248 (71%) + 분류층 514


세 팔을 실행합니다. 이 장난감 작업은 데이터가 충분하고 패턴이 단순해
**세 팔 모두 검증 100%** — 본문이 강조하는 대로 여기서의 포인트는 어느
쪽이 더 좋은가가 아니라, B(특징 추출)가 파라미터의 29%만 건드리고도
같은 결과를 낸다는 점입니다.

In [5]:
def make_data(n, size=16, period=4, width=2, noise=0.3, seed=1):
    """세로줄(0) vs 가로줄(1) 16×16 이미지 + 잡음. 10.2와 동일한 작업."""
    rng = np.random.RandomState(seed)
    pat = (np.arange(size) % period) < width
    Xv = np.tile(pat, (size, 1))   # 세로줄
    Xh = Xv.T.copy()               # 가로줄
    X, y = [], []
    for i in range(n):
        base = Xv if i % 2 == 0 else Xh
        X.append((base + rng.normal(0, noise, base.shape))[None])
        y.append(i % 2)
    return torch.tensor(np.stack(X).astype(np.float32)), torch.tensor(y)

Xtr, ytr = make_data(300)
Xva, yva = make_data(200, seed=2)

def train(arm, epochs=30):
    torch.manual_seed(42)
    model = SmallCNN()
    if arm == "frozen":      # B. 특징 추출: 합성곱 동결, fc만 학습
        for p in model.conv1.parameters(): p.requires_grad = False
        for p in model.conv2.parameters(): p.requires_grad = False
        lr = 0.01
    elif arm == "finetune":  # C. 미세조정: 전체 학습, 낮은 학습률
        lr = 0.001
    else:                    # A. 스크래치
        lr = 0.01
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    lossfn = nn.CrossEntropyLoss()
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 50):
            idx = perm[i:i+50]
            opt.zero_grad()
            loss = lossfn(model(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step()
    with torch.no_grad():
        val = (model(Xva).argmax(1) == yva).float().mean().item()
        tr  = (model(Xtr).argmax(1) == ytr).float().mean().item()
    return tr, val

for arm, name in [("scratch", "A. 스크래치 (전체 1,762)"),
                  ("frozen",  "B. 특징 추출 (fc 514만)"),
                  ("finetune","C. 미세조정 (1,762, lr 0.001)")]:
    tr, va = train(arm)
    print(f"{name}: train {tr:.0%} / val {va:.0%}")

A. 스크래치 (전체 1,762): train 100% / val 100%
B. 특징 추출 (fc 514만): train 100% / val 100%


C. 미세조정 (1,762, lr 0.001): train 100% / val 100%


## 4. 두 단계 전이: "다른" 작업에서 배운 특징을 가져오기

(1) **큰 데이터**(4,000장, period 3 줄무늬 — "ImageNet"의 위장)에서
사전학습합니다. (2) 그 합성곱 가중치를 복사하고 동결한 뒤, **작은
데이터**(300장, period 4 — *주파가 다른* 새 문제)로 헤드만 학습합니다.
동결된 합성곱이 학습 후에도 bit-for-bit 그대로인지 확인합니다.

In [6]:
# 1단계: 큰 데이터(4,000장, period 3)에서 사전학습
Xbig, ybig = make_data(4000, period=3, seed=3)

torch.manual_seed(7)
pre = SmallCNN()
opt = torch.optim.Adam(pre.parameters(), lr=0.01)
lossfn = nn.CrossEntropyLoss()
for ep in range(15):
    perm = torch.randperm(len(Xbig))
    for i in range(0, len(Xbig), 200):
        idx = perm[i:i+200]
        opt.zero_grad(); loss = lossfn(pre(Xbig[idx]), ybig[idx]); loss.backward(); opt.step()
bigacc = (pre(Xbig).argmax(1) == ybig).float().mean().item()
print(f"1단계 (큰 데이터, period 3): 학습 정확도 {bigacc:.0%}")
assert bigacc == 1.0

# 2단계: 가중치 복사 → 합성곱 동결 → 작은 데이터(period 4)로 헤드만 학습
model = SmallCNN()
with torch.no_grad():
    for d, s in zip(model.conv1.parameters(), pre.conv1.parameters()): d.copy_(s)
    for d, s in zip(model.conv2.parameters(), pre.conv2.parameters()): d.copy_(s)
for p in model.conv1.parameters(): p.requires_grad = False
for p in model.conv2.parameters(): p.requires_grad = False
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"2단계 동결 후 학습 가능한 파라미터: {n_trainable} (fc 층뿐)")
assert n_trainable == 514
opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)
for ep in range(20):
    perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), 50):
        idx = perm[i:i+50]
        opt.zero_grad(); loss = lossfn(model(Xtr[idx]), ytr[idx]); loss.backward(); opt.step()
va = (model(Xva).argmax(1) == yva).float().mean().item()
print(f"2단계 (작은 데이터, period 4, 헤드만 학습): 검증 정확도 {va:.0%}")

1단계 (큰 데이터, period 3): 학습 정확도 100%
2단계 동결 후 학습 가능한 파라미터: 514 (fc 층뿐)
2단계 (작은 데이터, period 4, 헤드만 학습): 검증 정확도 100%


동결을 코드 수준에서 검증합니다 — `requires_grad=False`로 고정한 층의
가중치는 `.backward()`를 20 에폭 동안 아무리 호출해도 단 한 번도
갱신되지 않습니다(본문 "실습" 절의 주장을 직접 확인).

In [7]:
conv_before = [p.detach().clone() for p in list(pre.conv1.parameters()) + list(pre.conv2.parameters())]
after = list(model.conv1.parameters()) + list(model.conv2.parameters())
for b, a in zip(conv_before, after):
    assert torch.equal(b, a)
print("동결 확인: 학습 후 conv1/conv2 가중치가 사전학습 모델과 완전히 동일")

동결 확인: 학습 후 conv1/conv2 가중치가 사전학습 모델과 완전히 동일


## 참고: 이 실험이 보여준 것과 안 보여준 것

세 팔 모두 100%에 도달한 것은 이 16×16 줄무늬 작업이 *너무* 쉽기
때문입니다 — 무작위 초기화 합성곱조차 세로줄/가로줄을 구분할 수
있어서, "사전학습"이 필연적인 이점을 내지 못합니다. 전이학습의
이점이 *극적으로* 드러나는 조건은 (1) 데이터가 적다, (2) 새 도메인의
패턴이 앞쪽 층이 배운 범용 특징(모서리·대비)을 실제로 활용할 수
있을 때이며, 본문 "어떤 데이터를 언제 쓸 것인가" 표가 이 판단
기준입니다. 이 노트북의 가치는 정확도 차이를 재는 데 있는 것이
아니라, **동결의 기계적 동작**(파라미터 장부 514/1,762, bit-for-bit
동일성)을 코드 수준에서 확인할 수 있게 하는 데 있습니다.